In [ ]:
import os, re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
# from pymatgen.ext.matproj import MPRester
from mp_api.client import MPRester
from pymatgen.core import Composition, Element
from pymatgen.analysis.phase_diagram import GrandPotentialPhaseDiagram, PhaseDiagram
from pymatgen.analysis.interface_reactions import InterfacialReactivity, GrandPotentialInterfacialReactivity
from emmet.core.thermo import ThermoType
from pymatgen.entries.computed_entries import ComputedEntry
from matplotlib.patches import Patch
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator
from pathlib import Path



In [ ]:
%%capture
import siman #program package to manage DFT calculations https://github.com/dimonaks/siman
from siman.calc_manage import smart_structure_read, get_structure_from_matproj, get_structure_from_matproj_new
from siman.calc_manage import add, res
# Update configurations
from siman import header
from siman.database import write_database, read_database
from siman.set_functions import read_vasp_sets
from siman.header import db
from siman.header import _update_configuration
_update_configuration('project_conf.py')
read_database() # read saved database if available
from pydoc import importfile
project_sets = importfile('project_sets.py')
varset = read_vasp_sets(project_sets.user_vasp_sets, override_global = 1) #read user sets

from siman import thermo

# header.PATH2PROJECT =phonons_setup_tight '../dft_calculations/'
header.PATH2EDITOR = 'notepad.exe'
header.PATH2POTENTIALS = "/home/a.burov/soft/vasp_potentials/potpaw_PBE_MPIE/"

from matplotlib import rc



In [ ]:
from pymatgen.io.vasp import Chgcar

In [ ]:
from pymatgen.analysis.defects.generators import ChargeInterstitialGenerator


In [ ]:
from matplotlib import font_manager as fm
font_path = "/home/a.burov/fonts/ARIAL.TTF"
fm.fontManager.addfont(font_path)


In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
np.set_printoptions(precision=5)


In [ ]:
# %matplotlib inline
plt.rcParams['figure.dpi'] = 450


In [ ]:
API_KEY = "LTSM6dStBrl69FjopxP7KdZBP35B1yh7"

In [ ]:
cig = ChargeInterstitialGenerator()

In [ ]:
chg_path = "/home/a.burov/icys_2025/niohf/layered/layered.sc.9bulk_eos/100.CHGCAR"

In [ ]:
chg = Chgcar.from_file(chg_path)
    

In [ ]:
defects = list(cig.generate(chg, insert_species=["Li"]))


In [ ]:
defects

[Li intersitial site at [0.07,0.62,0.47],
 Li intersitial site at [0.10,0.13,0.49]]

## H exchange for Li, Na, and K

Two different questions get mixed if every H is deleted and the empty cell is then treated as a defect host.

**Dilute exchange (one H replaced in a supercell).** This is a substitution on H, not an interstitial. `ChargeInterstitialGenerator` inserts Li into a charge-density minimum and leaves H in the crystal, so the product is intercalated Ni(OH)F:Li. `doped` is the better tool: it builds the supercell, the H→alkali substitutions, and the charge states, and the later `doped` thermodynamics use competing phases as chemical potentials.

```python
from doped.generation import DefectsGenerator

# Relaxed Ni(OH)F primitive cell. Do not start from a cell that already has every H removed.
gen = DefectsGenerator(
    host_structure,
    extrinsic={"H": ["Li", "Na", "K"]},
    vacancy_elements=[],
    interstitial_elements=[],
    neutral_only=True,
)
```

`extrinsic={"H": ["Li", "Na", "K"]}` substitutes only on hydrogen. The empty element lists turn off vacancies and interstitials, which `DefectsGenerator` would otherwise add. Formation energies then come from the `doped` parsing and thermodynamics workflow (https://doped.readthedocs.io/en/latest/index.html). A negative neutral formation energy, relative to the reservoir you choose for Li/Na/K and for the removed H, means that single exchange is favorable in the dilute limit.

**Complete exchange (every H replaced).** That is a new bulk crystal, Ni(OLi)F and the Na and K analogues, not a point defect. Delete H only as the starting geometry: put Li, Na, or K on those sites and relax. A single-point energy of the frozen, fully deprotonated cell is a poor host energy, because that cell is not the material you compare against. The exchange energy per proton is

`E[Ni(OA)F] + μ(H) − E[Ni(OH)F] − μ(A)`

with A = Li, Na, or K. Take μ(A) and μ(H) from the same reservoir (elemental reference, or the phase the ions are exchanged with). Rank the three alkalis by this energy after the ionic relaxation. Use the dilute `doped` calculation only when the question is whether a single H site exchanges, not whether the fully exchanged phase is stable.


In [ ]:
import sys
from pathlib import Path

def _repo_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "plot_colors.py").is_file():
            return cand
    raise FileNotFoundError("plot_colors.py not found from " + str(here))

sys.path.insert(0, str(_repo_root()))
from plot_colors import series_colors
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

fig, axes = plt.subplots(1, 3, figsize=(10, 1.6))
for ax, n in zip(axes, (2, 3, 6)):
    cols = series_colors(n)
    ax.imshow(np.arange(n)[None, :], cmap=ListedColormap(cols), aspect="auto")
    ax.set_title(f"n={n}")
    ax.set_xticks(range(n))
    ax.set_xticklabels(cols, rotation=45, fontsize=7)
    ax.set_yticks([])
fig.tight_layout()
plt.show()
